In [4]:
import pandas as pd

# 1. Load the two datasets
df_abund = pd.read_csv('/Users/yarintamam/Code/Ask-SLM/Workspaces/ABX/results/inputs/single_taxon_abundance/single_taxon_abundance_Viruses_raw.csv')
df_canon = pd.read_csv('/Users/yarintamam/Downloads/canon.csv')

# 2. Merge the datasets
# We match the 'sample_name' from the abundance file with the 'library' column in the canon file
merged = pd.merge(df_abund, df_canon, left_on='sample_name', right_on='library', how='inner')

# 3. Pivot the table
# - index='subject' makes the subjects the rows
# - columns='day' makes the days the columns
# - values='relative_abundance' fills the table with the abundance values
# - aggfunc='mean' ensures that if a subject has multiple readings on the exact same day, 
#   they are averaged into a single value, guaranteeing each subject/day combination appears only once.
pivot_df = merged.pivot_table(index='subject', columns='day', values='relative_abundance', aggfunc='mean')

# 4. Save the resulting table to a new CSV file
output_file = 'subject_day_virus_abundance.csv'
pivot_df.to_csv(output_file)

In [6]:

# Condense the rows by removing NaNs and shifting data to the left
condensed_data = []
subjects = []

for subject, row in pivot_df.iterrows():
    # dropna() removes all NaN values, keeping chronological order
    values = row.dropna().tolist()
    condensed_data.append(values)
    subjects.append(subject)

# Create a new DataFrame from the lists
condensed_df = pd.DataFrame(condensed_data, index=subjects)

# Rename the columns sequentially
condensed_df.columns = [f"time_point_{i+1}" for i in range(condensed_df.shape[1])]
condensed_df.index.name = 'subject'

# Save to a new CSV file
output_file = 'condensed_virus_abundance.csv'
condensed_df.to_csv(output_file)

In [4]:
import requests
import urllib.parse

def get_envo_id_from_api(term: str) -> str:
    # EBI OLS API endpoint for ENVO
    base_url = "https://www.ebi.ac.uk/ols4/api/search"
    query = urllib.parse.quote(term)

    # מחפשים בדיוק באונטולוגיה של ENVO
    url = f"{base_url}?q={query}&ontology=envo&queryFields=label,synonym&exact=false"

    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['response']['numFound'] > 0:
            # מחזיר את התוצאה הכי רלוונטית
            best_match = dawta['response']['docs'][0]
            return {"id": best_match['obo_id'], "name": best_match['label']}

    return {"error": "Term not found"}



get_envo_id_from_api("stool")

{'id': 'UBERON:0001988', 'name': 'feces'}

In [9]:
from ete3 import NCBITaxa

ncbi = NCBITaxa()

# 1. תרגום מ-ID לשם (הפונקציה מקבלת רשימת מספרים ומחזירה מילון)
taxid_to_name_dict = ncbi.get_taxid_translator([9606])
species_name = taxid_to_name_dict[9606]
print(species_name)  # פלט: Homo sapiens

# 2. תרגום משם ל-ID
# מעבירים את שם המין בתוך רשימה כפי שהפונקציה מצפה
name_to_taxid_dict = ncbi.get_name_translator([species_name])
print(name_to_taxid_dict)  # פלט: {'Homo sapiens': [9606]}

# 3. קבלת השושלת הטקסונומית (מחזיר רשימה של מזהים)
lineage_taxids = ncbi.get_lineage(9606)
print(lineage_taxids)  # פלט: [1, 131567, 2759, 33154, 33208, ...]

Homo sapiens
{'Homo sapiens': [9606]}
[1, 131567, 2759, 33154, 33208, 6072, 33213, 33511, 7711, 89593, 7742, 7776, 117570, 117571, 8287, 1338369, 32523, 32524, 40674, 32525, 9347, 1437010, 314146, 9443, 376913, 314293, 9526, 314295, 9604, 207598, 9605, 9606]


In [13]:
import sqlite3
from ete3 import NCBITaxa

ncbi = NCBITaxa()

def get_all_tax_variants(term_name: str) -> list[int]:
    """
    מקבל שם של טקסונומיה ומחזיר רשימה של ה-ID הנוכחי וכל ה-IDs הישנים שמוזגו אליו.
    """
    # 1. מציאת ה-ID הנוכחי והעדכני ביותר
    name_to_id = ncbi.get_name_translator([term_name])
    if not name_to_id:
        return []  # לא נמצא

    current_id = name_to_id[term_name][0]
    all_ids = [current_id]

    # 2. שליפת כל ה-IDs הישנים שמוזגו (Merged) אל ה-ID הנוכחי
    # אנחנו ניגשים ישירות ל-SQLite הפנימי ש-ete3 בנה
    conn = sqlite3.connect(ncbi.dbfile)
    cursor = conn.cursor()

    # טבלת merged מכילה: taxid_old, taxid_new
    cursor.execute("SELECT taxid_old FROM merged WHERE taxid_new = ?", (current_id,))
    old_ids = [row[0] for row in cursor.fetchall()]
    conn.close()

    # הוספת ה-IDs הישנים לרשימה
    all_ids.extend(old_ids)

    return all_ids

# דוגמה לשימוש (הפלט יהיה רשימה של מספרים)
ids_to_search = get_all_tax_variants("Ruminococcaceae")
print(ids_to_search)

[216572, 541000, 2304686]
